# Night-sky radiance and observation windows at CTAO sites with nsb-py

### Point evaluations and threshold-based planning

The optical night-sky background changes with viewing direction, time and observing
location. **nsb-py** combines airglow, zodiacal light, integrated starlight and
scattered moonlight to predict **photon radiance**, expressed in
**photons cm⁻² ns⁻¹ sr⁻¹**. This guide uses its public Python API to explore those
predictions at **CTAO South and CTAO North** over the default **300–650 nm** band.

The examples cover:

1. Defining an observer, a UTC instant and an ICRS sky direction.
2. Calculating total radiance and component contributions at both sites.
3. Comparing Airglow models over a range of zenith distances.
4. Finding intervals that satisfy a radiance threshold and observing constraints.

We use a fixed historical example: the **Crab Nebula**, on **15 January 2024**,
with a common solar input of **F10.7 = 130 sfu**. Parameters are editable so you can
adapt the workflow to another target or date.

The CTAO profiles provide **illustrative planning assumptions** and are not
measured site calibrations. Results describe sky radiance; camera counts and
instrument sensitivity require additional modeling. For observational comparisons with measured sky
spectra and nsb2, see [observational validation](observational-validation.ipynb).

Code: AGPL-3.0-only, Copyright (C) 2026 Vallés Puig, Ramon. Upstream data retain their source terms.


## 1. Prepare the Python environment

Select a fresh **CPython 3.12 or 3.13** Jupyter or Google Colab kernel and
**Run All**. The next cell installs **`nsb-py==0.2.0` from PyPI** and this
notebook's dependencies using the current kernel's Python. No NSB checkout,
GitHub installation of NSB or Rust toolchain is needed; a compatible NSB wheel
is required. The distribution is `nsb-py`, imported as `nsb`; the unrelated
package named `nsb` is rejected.

The package list is visible in `REQUIREMENTS`. Matching installed versions are
reused. If an already imported package changes, setup stops and asks you to
**restart the kernel and Run All**. Installation failures stop execution with
pip's error log; check network access and that you selected a supported kernel.

Results are generated inline using bundled NSB assets; no files are exported.


In [ ]:
# Run this cell first, using a CPython 3.12 or 3.13 kernel.
import importlib
import importlib.metadata as metadata
import json
from pathlib import Path
import subprocess
import sys

if sys.implementation.name != "cpython" or sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("Select a CPython 3.12 or 3.13 kernel, then Run All.")

# nsb-py is the distribution; nsb is its Python import. Do not install 'nsb'.
REQUIREMENTS = ['nsb-py==0.2.0', 'numpy>=2,<3', 'matplotlib>=3.8,<4', 'astropy>=6.1,<9']
try:
    metadata.distribution("nsb")
except metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("Remove the unrelated 'nsb' distribution with this kernel's pip, then restart.")

# Remember loaded dependencies so changes cannot silently leave stale imports.
loaded = {name: metadata.version(name)
          for module, names in metadata.packages_distributions().items()
          if module in sys.modules for name in names}
pip = [sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple"]
try:
    nsb_dist = metadata.distribution("nsb-py")
    direct = json.loads(nsb_dist.read_text("direct_url.json") or "{}")
except metadata.PackageNotFoundError:
    nsb_dist, direct = None, {}
if nsb_dist is not None and (nsb_dist.version != "0.2.0" or direct):
    if "nsb" in sys.modules:
        raise RuntimeError("Restart the kernel before selecting nsb-py==0.2.0 from PyPI.")
    subprocess.run(pip + ["--only-binary=:all:", "--no-deps", "--force-reinstall", "nsb-py==0.2.0"], check=True)
subprocess.run(pip + ["--only-binary=nsb-py", *REQUIREMENTS], check=True)

importlib.invalidate_caches()
changed = [name for name, version in loaded.items() if metadata.version(name) != version]
if changed:
    raise RuntimeError(f"Updated already-imported packages: {changed}. Restart the kernel, then Run All.")

# Check that the module resolves to the nsb-py installation, rather than a local file.
nsb_dist = metadata.distribution("nsb-py")
nsb_spec = importlib.util.find_spec("nsb")
owned = {Path(nsb_dist.locate_file(f)).resolve() for f in nsb_dist.files or []}
if nsb_dist.version != "0.2.0" or nsb_spec is None or not nsb_spec.origin or Path(nsb_spec.origin).resolve() not in owned:
    raise RuntimeError("Expected the nsb-py==0.2.0 module. Remove shadowing nsb.py/directories and restart.")
print("Ready: nsb-py", nsb_dist.version)


In [ ]:
from datetime import datetime, timezone

import astropy.units as u
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time
from astropy.utils import iers
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
import nsb
import siderust

# Use the Earth-orientation tables bundled with Astropy, without downloading updates.
iers.conf.auto_download = False


### Observing locations and shared inputs

The coordinates below are the **WGS84 array positions in the project's observatory
catalog**. South is the CTAO array location, distinct from Paranal. Heights are
ellipsoidal heights, not the nominal elevations above sea level. A site profile
sets atmospheric planning assumptions independently of the observer coordinates.

| Site | Longitude [° east] | Latitude [° north] | WGS84 height [m] |
|---|---:|---:|---:|
| CTAO South | −70.31634444444444 | −24.683427777777776 | 2184.6 |
| CTAO North | −17.892005 | 28.762164 | 2240.2 |

The first comparison uses the same target and UTC instant at both sites. Its local
altitude differs between sites, so the results reflect both viewing geometry and
the selected profile; this is not a ranking of site quality.


In [ ]:
instant = datetime(2024, 1, 15, 2, tzinfo=timezone.utc)
solar_flux_sfu = 130.0
unit = "photons cm⁻² ns⁻¹ sr⁻¹"
sites = {
    "CTAO South": (-70.31634444444444, -24.683427777777776, 2184.6, nsb.SiteProfile.CTA_SOUTH),
    "CTAO North": (-17.892005, 28.762164, 2240.2, nsb.SiteProfile.CTA_NORTH),
}


## 2. Evaluate one sky direction

Start with **CTAO South** and the Crab Nebula at **02:00 UTC**. `Observer` supplies
the location, `Direction` supplies ICRS right ascension and declination in degrees,
and `NsbModelConfig` selects the site profile and solar input. Reuse the resulting
`NsbEvaluator` for multiple queries.

`evaluate` returns the total photon radiance and its component contributions.
Here all four components are included in the default 300–650 nm band. The
model estimates the background **toward** the target; it does not add the Crab
Nebula's own source emission to that background.


In [ ]:
observer = siderust.Observer(lon_deg=-70.31634444444444, lat_deg=-24.683427777777776, height_m=2184.6)
target = siderust.Direction(ra_deg=83.6331, dec_deg=22.0145)  # Crab Nebula, ICRS
config = (nsb.NsbModelConfig.generic_clear_sky()
          .with_site_profile(nsb.SiteProfile.CTA_SOUTH)
          .with_solar_radio_flux_sfu(solar_flux_sfu))
evaluator = nsb.NsbEvaluator(config)
query = nsb.PointQuery(observer, instant, target)
result = evaluator.evaluate(query)

print(f"CTAO South, {instant.isoformat()}, 300–650 nm")
print(f"Total: {result.integrated_photons_cm2_ns_sr:.5f} {unit}")
for component in result.components:
    print(f"  {component.name}: {component.integrated_photons_cm2_ns_sr:.5f} {unit}")


## 3. Compare the component budgets at both sites

Evaluate the same target at each location. The table gives the **total** and each
component in radiance units; the bars compare the source contributions directly.
A larger contribution means that source predicts a brighter sky in the specified
direction and conditions. Moonlight follows the actual lunar geometry at this
instant; a zero contribution should not be interpreted as a general site property.


In [ ]:
observers, evaluators, results = {}, {}, {}
for name, (lon, lat, height, profile) in sites.items():
    observers[name] = siderust.Observer(lon_deg=lon, lat_deg=lat, height_m=height)
    config = (nsb.NsbModelConfig.generic_clear_sky()
              .with_site_profile(profile).with_solar_radio_flux_sfu(solar_flux_sfu))
    evaluators[name] = nsb.NsbEvaluator(config)
    results[name] = evaluators[name].evaluate(nsb.PointQuery(observers[name], instant, target))

components = {name: {c.name: c.integrated_photons_cm2_ns_sr for c in r.components}
              for name, r in results.items()}
component_names = list(components["CTAO South"])
table = "| Site | Total | " + " | ".join(component_names) + " |\n"
table += "|---|" + "---:|" * (len(component_names) + 1) + "\n"
for name, r in results.items():
    table += f"| {name} | {r.integrated_photons_cm2_ns_sr:.5f} | "
    table += " | ".join(f"{components[name][c]:.5f}" for c in component_names) + " |\n"
display(Markdown(table + f"\nAll values: {unit}; 300–650 nm."))

fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
x = np.arange(len(component_names))
for index, name in enumerate(sites):
    ax.bar(x + (index - 0.5) * 0.36, [components[name][c] for c in component_names], width=0.36, label=name)
ax.set(xticks=x, xticklabels=component_names, ylabel=unit, title="Crab direction: sky components at 02:00 UTC, 15 January 2024")
ax.legend()
plt.show()


## 4. Explore Airglow versus zenith distance

In v0.2.0, **Automatic selects experimental PALACE Oxygen**: continuum plus Oxygen
emission. Selecting **PALACE continuum only** changes the physical content of the
prediction. The comparison below holds the site, UTC instant and F10.7 input fixed
while changing viewing direction and Airglow model.

These are local-north directions (azimuth 0°), from zenith to **59° zenith distance**,
rather than a time track of the Crab Nebula. The small coordinate helper converts
each local direction into the ICRS type accepted by `PointQuery`. Zero atmospheric
pressure in that transformation disables apparent refraction.

Both panels show **Airglow only**, integrated over 300–650 nm. The separation of
the curves shows the modeled Oxygen contribution. Automatic/Oxygen requires the
standard continuum geometry and **zenith distance ≤60°**. The Paranal-derived
model remains a spatial extrapolation at CTAO sites; stay inside this domain when
editing directions or observing windows. Continuum is an explicit alternative
with different physical assumptions, not an automatic fallback.


In [ ]:
def local_direction(location, instant, zenith_distance_deg):
    sky = SkyCoord(az=0*u.deg, alt=(90-zenith_distance_deg)*u.deg,
                   frame=AltAz(obstime=Time(instant), location=location, pressure=0*u.hPa)).icrs
    return siderust.Direction(ra_deg=float(sky.ra.deg), dec_deg=float(sky.dec.deg))


zenith_distances = np.linspace(0, 59, 16)
airglow_curves = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained", sharey=True)
for ax, (name, (lon, lat, height, profile)) in zip(axes, sites.items()):
    location = EarthLocation.from_geodetic(lon*u.deg, lat*u.deg, height*u.m)
    models = {
        "Automatic (Oxygen)": evaluators[name],
        "Continuum only": nsb.NsbEvaluator(evaluators[name].config.with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)),
    }
    airglow_curves[name] = {}
    for label, model in models.items():
        values = [model.integrate_band(nsb.PointQuery(
            observers[name], instant, local_direction(location, instant, float(zd)),
            components=nsb.ComponentMask.AIRGLOW), 300, 650) for zd in zenith_distances]
        airglow_curves[name][label] = values
        ax.plot(zenith_distances, values, label=label)
    ax.set(title=name, xlabel="Zenith distance [degrees]", ylabel=unit)
    ax.legend()
fig.suptitle("Airglow, 300–650 nm; F10.7 = 130 sfu; 02:00 UTC, 15 January 2024")
plt.show()


## 5. Find intervals below a sky-radiance threshold

Keep the same target and evaluators, and search **01:15–03:00 UTC on 15 January
2024**. `ThresholdQuery` defines the search window, radiance ceiling and observing
constraints; `periods_below_threshold` returns pairs of UTC datetimes.

The settings below require:

- **Total sky radiance:** below **0.35 photons cm⁻² ns⁻¹ sr⁻¹**, in the default 300–650 nm band.
- **Astronomical night:** Sun altitude below −18°.
- **Target altitude:** at least 30°, consistent with the Oxygen zenith-distance domain.

The target stays within the supported domain throughout this example's search
window. Changing dates or extending the window can move it outside that domain.
The radiance ceiling is an **illustrative user choice**, not a calibrated camera
limit or an instrument sensitivity requirement. A search can legitimately return
no intervals; this means no period satisfies the supplied settings.


In [ ]:
start = datetime(2024, 1, 15, 1, 15, tzinfo=timezone.utc)
end = datetime(2024, 1, 15, 3, tzinfo=timezone.utc)
threshold = 0.35  # Illustrative total radiance ceiling, in photons cm⁻² ns⁻¹ sr⁻¹
periods = {}
table = "| Site | Start [UTC] | End [UTC] |\n|---|---|---|\n"
for name, model in evaluators.items():
    query = nsb.ThresholdQuery(observers[name], target, start, end, threshold,
        components=nsb.ComponentMask.ALL, sun_altitude_ceiling_deg=-18.0, target_altitude_floor_deg=30.0)
    periods[name] = model.periods_below_threshold(query).periods
    for begin, finish in periods[name]:
        table += f"| {name} | {begin:%Y-%m-%d %H:%M:%S.%f} | {finish:%Y-%m-%d %H:%M:%S.%f} |\n"
    if not periods[name]:
        table += f"| {name} | No intervals satisfy these settings | Not applicable |\n"
display(Markdown(table))


### Read the returned intervals against the radiance curve

Each panel shows the predicted total sky radiance toward the fixed Crab direction.
The dashed line is the chosen ceiling; shaded spans are the intervals returned by
the planner, including its Sun and target-altitude constraints. A whole-window
interval is also a valid outcome. These panels use a common radiance scale.

The sampled curve is **for visualization only**. Interval boundaries come from the
API's adaptive search, independently of the plotting samples; that search follows
the model's documented empirical resolution assumptions rather than a mathematical
certification of every unvisited instant.

Abrupt changes can occur when modeled moonlight switches off at the lunar horizon.
Lines between plotting samples do not resolve such transitions; use the returned
UTC intervals for their boundaries.


In [ ]:
times = [start + (end-start)*i/60 for i in range(61)]
radiance_tracks = {}
fig, axes = plt.subplots(2, 1, figsize=(11, 7), layout="constrained", sharex=True, sharey=True)
for ax, (name, model) in zip(axes, evaluators.items()):
    values = [model.evaluate(nsb.PointQuery(observers[name], t, target)).integrated_photons_cm2_ns_sr for t in times]
    radiance_tracks[name] = values
    ax.plot(times, values, label="Predicted total radiance")
    ax.axhline(threshold, color="black", linestyle="--", label=f"Ceiling: {threshold:.2f}")
    for index, (begin, finish) in enumerate(periods[name]):
        ax.axvspan(begin, finish, color="tab:green", alpha=0.18, label="Returned interval" if index == 0 else None)
    ax.set(title=name, ylabel=unit, xlim=(start, end))
    ax.legend(fontsize=9)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=timezone.utc))
axes[-1].set_xlabel("Time [UTC]")
fig.suptitle(f"Crab direction: illustrative observation windows, {start:%d %B %Y}")
plt.show()


## 6. Adapt and interpret the example

For another target or date, edit the observer coordinates and profile, the ICRS target,
the timezone-aware UTC dates, the solar input and the radiance threshold. Re-run
from a fresh kernel so the point queries, curves and planning windows use the same
configuration. Keep the target within the selected Airglow model's domain.

The component plots describe a modeled radiance budget; the Airglow curves isolate
a model choice; the returned intervals apply that model to explicit planning
criteria. Their interpretation is conditional on those inputs and on the
experimental Airglow and site-profile assumptions. They do not establish CTAO
observational calibration or a prediction of detector performance. For evidence
about agreement with observations, use [observational validation](observational-validation.ipynb) and the
[scientific validation guide](../docs/scientific-validation.md).
